# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import pandas as pd
PATH = '/content/drive/MyDrive/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [13]:
# TODO (hint: .str.lower().str.contains('pop'))

pop_rows = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]

print(f"Count: {len(pop_rows)}")
display(pop_rows)

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [14]:
is_strawberry = pop_rows['product_name'].str.contains(
    'strawberry', case=False, na=False
)

strawberry_pop = pop_rows[is_strawberry]
other_pop = pop_rows[~is_strawberry]

display(strawberry_pop[['sku', 'product_name']])

print("Non-strawberry Pop-Tarts:")
display(other_pop[['sku', 'product_name']])

,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Non-strawberry Pop-Tarts:


,sku,product_name


**Pop-Tart rows that are a different product:** _..._

All four refer to strawberry Pop-Tarts. However, in the full catalog, it's important to look for other flavors containing "Pop-Tart" that is not strawberry. Any other flavor would stay separate.


### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [15]:
sku_mapping = {
    'PT-12': 'SKU#459812',
    'POPTART-STRAW': 'SKU#459812',
    'PT_STRAWBERRY': 'SKU#459812'
}

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

display(cat.loc[pop_rows.index, ['sku', 'canonical_sku', 'product_name']])

,sku,canonical_sku,product_name
0,PT-12,SKU#459812,pop-tarts strawberry
1,SKU#459812,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,SKU#459812,Pop-Tarts Strawberry
3,PT_STRAWBERRY,SKU#459812,pop-tarts strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [16]:
print("Before")
display(cat.loc[pop_rows.index, 'category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()

print("After")
display(cat.loc[pop_rows.index, 'category'].value_counts(dropna=False))

display(cat.loc[pop_rows.index, ['product_name', 'category']])

Before


,count
category,
Breakfast,2
Breakfast Foods,1
breakfast,1


After


,count
category,
Breakfast,3
Breakfast Foods,1


,product_name,category
0,pop-tarts strawberry,Breakfast Foods
1,Strawberry Pop-Tart,Breakfast
2,Pop-Tarts Strawberry,Breakfast
3,pop-tarts strawberry,Breakfast


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [17]:
cat['normalized'] = (
    cat['product_name'].str.lower()
    .str.replace(r'[^a-z ]', '', regex=True)
    .str.split()
    .apply(lambda w: ' '.join(sorted(w)))
)

candidates = (
    cat.groupby('normalized')['sku']
    .agg(lambda s: list(s.unique()))
    .loc[lambda s: s.str.len() > 1]
)

display(candidates)

,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [18]:
rows_before = len(cat)

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

assert cat['canonical_sku'].isna().sum() == 0, "Found null canonical SKUs!"
assert len(cat) == rows_before, "Row count changed!"

print("Checks passed: no null canonical SKUs and row count unchanged.")

Checks passed: no null canonical SKUs and row count unchanged.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families that appear fragmented are:
1. Bottled Water: Variants like "Bottled Water 24pk", "bottled water 24 pack", and "WTR-24" all describe the same item but use different text casing and packaging phrasing. I would group them by normalizing the text string and making sure they are under the canonical SKU WTR-24, since their names all describe 24-pack water.
2. AA Batteries: "Flashlight AA", "BAT-AA4", and "BATT_AA_4PK" appear to describe four-packs of AA batteries. After checking the conflicting package sizes, I would map them to BAT-AA4 and standardize their category to “Emergency Supplies.” Other battery types or confirmed different pack sizes would stay separate.

b) Consolidation stops being a data-cleaning task and becomes a business decision when it requires deciding whether product differences matter for inventory or reporting. For example, STREAM-01 is named “Roku Streaming Stick,” while ROKU-STD is only “Streaming Device.” I am not sure whether these are the same brand and model or different devices. I would ask the electronics category manager to confirm before merging them.